# Avro Basics — 04: Schema Evolution


Avro's schema evolution is the feature that makes it ideal for Kafka pipelines
— producers and consumers can evolve schemas independently.

Topics: backward/forward/full compatibility, adding fields with defaults,
removing fields, using aliases for renames, compatibility rules.


In [1]:
import os, time, pathlib, shutil, random, datetime, json as pyjson
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/avro_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('avro-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version} | Avro JAR: spark-avro_2.13-4.0.2")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/09 13:09:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.0.2 | Avro JAR: spark-avro_2.13-4.0.2


## Step 1 — Backward Compatibility: New Reader, Old Data

Add field WITH default → old data readable by new schema.

In [2]:

import json as pyjson, pathlib

# V1 schema
SCHEMA_V1 = pyjson.dumps({"type":"record","name":"Event","fields":[
    {"name":"event_id",  "type":"long"},
    {"name":"user_id",   "type":"long"},
    {"name":"event_type","type":"string"},
]})

# V2 schema: added fields WITH DEFAULTS (backward compatible)
SCHEMA_V2 = pyjson.dumps({"type":"record","name":"Event","fields":[
    {"name":"event_id",  "type":"long"},
    {"name":"user_id",   "type":"long"},
    {"name":"event_type","type":"string"},
    {"name":"device",    "type":"string",         "default":"unknown"},  # NEW with default
    {"name":"revenue",   "type":["null","double"], "default":None},      # NEW nullable
]})

v1_data = spark.createDataFrame([(1,101,"click"),(2,102,"purchase")], ["event_id","user_id","event_type"])
v1_path = f"{DATA_DIR}/evo_v1"
v1_data.write.format("avro").option("avroSchema", SCHEMA_V1).mode("overwrite").save(v1_path)
print("V1 data written with V1 schema")

# Read V1 data with V2 schema (backward compatibility)
df_v1_as_v2 = spark.read.format("avro").option("avroSchema", SCHEMA_V2).load(v1_path)
print("\nV1 data read with V2 schema (new fields = defaults):")
df_v1_as_v2.show(truncate=False)
print("✅ device='unknown', revenue=null  ← defaults applied for missing fields")


V1 data written with V1 schema

V1 data read with V2 schema (new fields = defaults):
+--------+-------+----------+-------+-------+
|event_id|user_id|event_type|device |revenue|
+--------+-------+----------+-------+-------+
|2       |102    |purchase  |unknown|NULL   |
|1       |101    |click     |unknown|NULL   |
+--------+-------+----------+-------+-------+

✅ device='unknown', revenue=null  ← defaults applied for missing fields


## Step 2 — Forward Compatibility: Old Reader, New Data

New data has extra fields — old reader ignores them.

In [3]:

v2_data = spark.createDataFrame([
    (3, 103, "purchase", "mobile", 99.99),
    (4, 104, "view",     "desktop", None),
], ["event_id","user_id","event_type","device","revenue"])
v2_path = f"{DATA_DIR}/evo_v2"
v2_data.write.format("avro").option("avroSchema", SCHEMA_V2).mode("overwrite").save(v2_path)
print("V2 data written")

# Read V2 data with V1 schema (forward compat — extra fields ignored)
df_v2_as_v1 = spark.read.format("avro").option("avroSchema", SCHEMA_V1).load(v2_path)
print("\nV2 data read with V1 schema (extra fields ignored):")
df_v2_as_v1.show(truncate=False)
print("✅ device and revenue columns ignored — V1 reader only sees V1 fields")


V2 data written

V2 data read with V1 schema (extra fields ignored):
+--------+-------+----------+
|event_id|user_id|event_type|
+--------+-------+----------+
|3       |103    |purchase  |
|4       |104    |view      |
+--------+-------+----------+

✅ device and revenue columns ignored — V1 reader only sees V1 fields


## Step 3 — Full Compatibility: Mixed Versions

Read both V1 and V2 files with the widest schema.

In [4]:

# Full compatibility: read both V1 and V2 files together
all_events = spark.read.format("avro").option("avroSchema", SCHEMA_V2).load(v1_path, v2_path)
print(f"Combined V1+V2 data with V2 schema ({all_events.count()} rows):")
all_events.show(truncate=False)
print("V1 rows have default values; V2 rows have real values — seamless ✅")


Py4JJavaError: An error occurred while calling o113.load.
: org.apache.spark.SparkClassNotFoundException: [DATA_SOURCE_NOT_FOUND] Failed to find the data source: /workspace/data/avro_basics/evo_v2. Make sure the provider name is correct and the package is properly registered and compatible with your Spark version. SQLSTATE: 42K02
	at org.apache.spark.sql.errors.QueryExecutionErrors$.dataSourceNotFoundError(QueryExecutionErrors.scala:722)
	at org.apache.spark.sql.execution.datasources.DataSource$.lookupDataSource(DataSource.scala:681)
	at org.apache.spark.sql.execution.datasources.DataSource$.lookupDataSourceV2(DataSource.scala:740)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource$$anonfun$apply$1.applyOrElse(ResolveDataSource.scala:58)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource$$anonfun$apply$1.applyOrElse(ResolveDataSource.scala:45)
	at org.apache.spark.sql.catalyst.plans.logical.AnalysisHelper.$anonfun$resolveOperatorsUpWithPruning$3(AnalysisHelper.scala

## Step 4 — Rename with Aliases

Aliases let old data map to a renamed field.

In [ ]:

# Rename event_type → event_name using alias
SCHEMA_V3 = pyjson.dumps({"type":"record","name":"Event","fields":[
    {"name":"event_id",   "type":"long"},
    {"name":"user_id",    "type":"long"},
    {"name":"event_name", "type":"string", "aliases":["event_type"]},  # renamed!
    {"name":"device",     "type":"string", "default":"unknown"},
    {"name":"revenue",    "type":["null","double"], "default":None},
]})

# V1 data (has event_type) readable with V3 schema (has event_name + alias)
df_v1_v3 = spark.read.format("avro").option("avroSchema", SCHEMA_V3).load(v1_path)
print("V1 data (event_type) read with V3 schema (event_name + alias):")
df_v1_v3.show(truncate=False)
print("✅ event_type mapped to event_name via alias")


## Summary



In [ ]:

print("""
Compatibility rules:
  BACKWARD : new schema reads OLD data
             → add fields WITH defaults
             → remove fields

  FORWARD  : old schema reads NEW data
             → remove fields WITH defaults
             → add fields

  FULL     : both directions work
             → only add/remove fields WITH defaults

Rules for safe evolution:
  ✅ Add field with default value
  ✅ Remove field that has a default value
  ✅ Add alias to renamed field
  ❌ Remove field without default
  ❌ Add field without default
  ❌ Change field type (without alias)
  ❌ Rename field without alias
""")
